# Databricks Notebook: 04_enrich_countries_data.ipynb

# Este notebook enriquece os dados de países processados com indicadores econômicos e os salva como uma tabela Delta no DBFS.

In [0]:
import logging
from typing import Dict, List, Optional, Any
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, IntegerType
)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

class EnrichmentError(Exception):
    """Custom exception for data enrichment errors."""
    pass

def create_spark_session(app_name: str) -> SparkSession:
    """
    Creates and returns a SparkSession with Delta Lake support.

    Args:
        app_name (str): The name of the Spark application.

    Returns:
        SparkSession: The configured SparkSession.
    """
    logger.info(f"Creating SparkSession for application: {app_name}")
    spark = SparkSession.builder \
        .appName(app_name) \
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
        .getOrCreate()
    logger.info("SparkSession created successfully.")
    return spark

def enrich_countries_data(spark: SparkSession, processed_countries_path: str, economic_data_path: str, output_path: str) -> None:
    """
    Enriches processed countries data with economic indicators and saves to Delta Lake.

    Args:
        spark (SparkSession): The active SparkSession.
        processed_countries_path (str): Path to the processed countries Delta table on DBFS.
        economic_data_path (str): Path to the economic data Delta table on DBFS.
        output_path (str): Path to save the enriched Delta table on DBFS.
    """
    logger.info(f"Reading processed countries data from {processed_countries_path}")
    try:
        countries_df = spark.read.format("delta").load(processed_countries_path)
        logger.info("Processed countries data read successfully.")
    except Exception as e:
        logger.error(f"Error reading processed countries data: {e}")
        raise EnrichmentError(f"Failed to read processed countries data: {e}")

    logger.info(f"Reading economic data from {economic_data_path}")
    try:
        economic_df = spark.read.format("delta").load(economic_data_path)
        logger.info("Economic data read successfully.")
    except Exception as e:
        logger.error(f"Error reading economic data: {e}")
        raise EnrichmentError(f"Failed to read economic data: {e}")

    logger.info("Joining dataframes and calculating additional metrics.")
    # Join on a common key, e.g., country code (cca3)
    # Assuming 'cca3' exists in both dataframes for joining
    enriched_df = countries_df.join(economic_df, on="cca3", how="left_outer") \
        .withColumn("gdp_per_capita", F.col("gdp_per_capita")) \
        .withColumn("hdi_score", F.col("hdi")) \
        .drop("gdp_per_capita", "hdi") # Drop original columns if renamed

    logger.info(f"Writing enriched data to Delta Lake at {output_path}")
    try:
        enriched_df.write.mode("overwrite").saveAsTable("countries_info")
        enriched_df.write.format("delta").mode("overwrite").save(output_path)
        logger.info("Enriched data successfully saved to Delta Lake.")
    except Exception as e:
        logger.error(f"Error saving enriched data to Delta Lake: {e}")
        raise EnrichmentError(f"Failed to save enriched data: {e}")

# Recebe os caminhos dos arquivos de entrada do notebook orquestrador
processed_countries_path = "/Volumes/workspace/default/data/processed/processed_countries"
economic_data_path = "/Volumes/workspace/default/data/simulated/economic_data"
output_path = "/Volumes/workspace/default/data/enriched/enriched_countries"

spark = create_spark_session("CountryDataEnrichment")
enrich_countries_data(spark, processed_countries_path, economic_data_path, output_path)
spark.stop()

# Retorna o caminho de saída para o notebook orquestrador
dbutils.notebook.exit(output_path)
